In [ ]:
import jax
import jax.numpy as jnp
from transformers import AutoProcessor, AutoTokenizer
from tunix.generate import sampler
from tunix.models.gemma4 import model
from tunix.models.gemma4 import params_safetensors

MESH = [(1, 2), ("fsdp", "tp")]
mesh = jax.make_mesh(
    *MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0])
)

# Download HF safetensors to your local dir.
# e.g. hf download google/gemma-4-31B-it --local-dir <your local path>
E2B_PATH = "<your local path>"
E4B_PATH = "<your local path>"
MOE_26B_PATH = "<your local path>"
DENSE_31B_PATH = "<your local path>"

version = "e2b"
if version == "e2b":
  tokenizer = AutoTokenizer.from_pretrained(E2B_PATH)
  config = model.ModelConfig.gemma4_e2b()
  model = params_safetensors.create_model_from_safe_tensors(
      E2B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=False
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=35, num_kv_heads=1, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(E2B_PATH)
elif version == "e4b":
  tokenizer = AutoTokenizer.from_pretrained(E4B_PATH)
  config = model.ModelConfig.gemma4_e4b()
  model = params_safetensors.create_model_from_safe_tensors(
      E4B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=False
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=42, num_kv_heads=2, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(E4B_PATH)
elif version == "moe_26b":
  tokenizer = AutoTokenizer.from_pretrained(MOE_26B_PATH)
  config = model.ModelConfig.gemma4_26b_a4b()
  model = params_safetensors.create_model_from_safe_tensors(
      MOE_26B_PATH, config, mesh, dtype=jnp.bfloat16, text_only=False
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=30, num_kv_heads=8, head_dim=512
  )
  processor = AutoProcessor.from_pretrained(MOE_26B_PATH)
elif version == "31b":
  tokenizer = AutoTokenizer.from_pretrained(DENSE_31B_PATH)
  config = model.ModelConfig.gemma4_31b()
  model = params_safetensors.create_model_from_safe_tensors(
      DENSE_31B_PATH, config, mesh, dtype=jnp.bfloat16
  )
  cache_config = sampler.CacheConfig(
      cache_size=2048, num_layers=60, num_kv_heads=16, head_dim=512
  )
else:
  raise ValueError(f"Unknown version: {version}")

In [ ]:
def templatize(prompts):
  out = []
  for p in prompts:
    out.append(
        tokenizer.apply_chat_template(
            [
                {"role": "user", "content": p},
            ],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )
  return out


inputs = templatize([
    "which is larger 9.9 or 9.11?",
    "why sky is blue?",
    "write code to solve fibonacci in python",
    "Write a short joke about saving RAM.",
])

sampler = sampler.Sampler(
    model,
    tokenizer,
    cache_config=cache_config,
)
with mesh:
  out = sampler(
      inputs,
      max_generation_steps=1024,
      echo=True,
      temperature=0.6,
      eos_tokens=[1, 106, 50],
  )

  for t in out.text:
    print(t)
    print("*" * 30)

## Vision example

import tensorflow_datasets as tfds

ds = tfds.data_source('oxford_flowers102', split='train')
image1 = ds[0]['image']
image2 = ds[1]['image']
image3 = ds[2]['image']

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(image1)

In [ ]:
plt.imshow(image2)

In [ ]:
messages = [
    [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the images."},
            {"type": "image"},
            {"type": "image"},
        ],
    }],
    [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Write a poem for the image."},
            {"type": "image"},
        ],
    }],
]

prompt = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

with mesh:
  output = sampler(
      input_strings=prompt,
      images=[[image1, image2], [image3]],
      max_generation_steps=128,
      temperature=1,
      echo=False,
      eos_tokens=[1, 106, 50],
  )
for o in output.text:
  print(o)
  print("####" * 8)

## Audio example

In [ ]:
import librosa
import numpy as np


def load_audio(path):
  try:
    # librosa automatically resamples to 16kHz and converts to float32 [-1.0, 1.0]
    y, _ = librosa.load(path, sr=16000)
    return y
  except Exception as e:
    print(f"Failed to load with librosa, trying scipy: {e}")
    from scipy.io import wavfile

    sr, y = wavfile.read(path)
    if y.ndim > 1:
      y = y.mean(axis=-1)
    if y.dtype == np.int16:
      y = y.astype(np.float32) / 32768.0
    elif y.dtype == np.int32:
      y = y.astype(np.float32) / 2147483648.0
    # Note: simple scipy loading does not resample. Ensure your wav is 16kHz.
    return y


# Example usage with real files (uncomment and replace with your paths):
# audio1 = load_audio("path/to/audio1.wav")
# audio2 = load_audio("path/to/audio2.wav")
# audio3 = load_audio("path/to/audio3.wav")

# Fallback to mock audio (zeros) for quick verification:
audio1 = jnp.zeros((16000 * 2,), dtype=jnp.float32)  # 2 seconds of silence
audio2 = jnp.zeros((16000 * 1,), dtype=jnp.float32)  # 1 second of silence
audio3 = jnp.zeros((16000 * 3,), dtype=jnp.float32)  # 3 seconds of silence

# Define messages with audio placeholders
audio_messages = [
    [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe these audio clips."},
            {"type": "audio"},
            {"type": "audio"},
        ],
    }],
    [{
        "role": "user",
        "content": [
            {"type": "text", "text": "What is in this audio?"},
            {"type": "audio"},
        ],
    }],
]

audio_prompt = processor.apply_chat_template(
    audio_messages, tokenize=False, add_generation_prompt=True
)

with mesh:
  audio_output = sampler(
      input_strings=audio_prompt,
      audios=[[audio1, audio2], [audio3]],
      max_generation_steps=128,
      temperature=1,
      echo=False,
      eos_tokens=[1, 106, 50],
  )

for o in audio_output.text:
  print(o)
  print("####" * 8)